# Pipeline de développement d’un modèle d’apprentissage automatique

## Objectif

Ce notebook transforme le cours sur le **pipeline de Machine Learning** en une série d’étapes pédagogiques.

Le principe général est :

**Données → Nettoyage → Transformation → Prétraitement → Séparation → Hyperparamètres → Entraînement → Évaluation → Déploiement → Surveillance**

> Les exemples de code sont volontairement génériques. Ils constituent une base de travail à adapter au jeu de données réel.

# Étape 1 — Importation des bibliothèques

Avant de commencer, on importe les bibliothèques Python nécessaires à l’analyse et à la construction des modèles.

Les bibliothèques couramment utilisées sont notamment :
- `pandas` pour manipuler les données ;
- `numpy` pour les calculs numériques ;
- `matplotlib` pour les graphiques ;
- `scikit-learn` pour le Machine Learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Bibliothèques importées avec succès.")

# Étape 2 — Charger les données

La première étape pratique consiste à charger le jeu de données.

Dans un projet réel, les données peuvent provenir d’un fichier CSV, d’une base de données, d’une API ou d’une autre source.

Exemple avec un fichier CSV.

In [ ]:
# Remplacer le nom du fichier par votre propre fichier
# df = pd.read_csv("mon_dataset.csv")

# Exemple pédagogique :
df = pd.DataFrame({
    "age": [25, 32, 41, 28, 36],
    "revenu": [25000, 42000, 55000, 31000, 48000],
    "achat": [0, 1, 1, 0, 1]
})

df.head()

# Étape 3 — Comprendre les données

Avant de modifier les données, il faut comprendre leur structure.

On peut examiner :
- le nombre de lignes et de colonnes ;
- les types de variables ;
- les premières observations ;
- les statistiques descriptives.

In [ ]:
print("Dimensions :", df.shape)
print("\nTypes de données :")
print(df.dtypes)

print("\nStatistiques descriptives :")
display(df.describe())

# Étape 4 — Nettoyage des données

Le nettoyage consiste notamment à rechercher :
- les valeurs manquantes (`NaN`) ;
- les doublons ;
- les données corrompues ou incohérentes.

Il faut comprendre la cause d’une anomalie avant de décider de la supprimer ou de la corriger.

In [ ]:
# Nombre de valeurs manquantes par colonne
print("Valeurs manquantes :")
print(df.isnull().sum())

# Nombre de lignes dupliquées
print("\nNombre de doublons :", df.duplicated().sum())

# Suppression des doublons si cela est justifié
df = df.drop_duplicates()

print("\nDimensions après nettoyage :", df.shape)

# Étape 5 — Transformation et représentation des données

Les modèles de Machine Learning travaillent principalement avec des représentations numériques.

Selon le type de données, il peut être nécessaire de transformer :
- du texte ;
- des catégories ;
- des images ;
- du son ;
- des séries temporelles.

La représentation choisie doit être adaptée au problème.

In [ ]:
# Exemple : séparation des variables explicatives et de la cible
X = df.drop(columns=["achat"])
y = df["achat"]

print("Variables explicatives :")
display(X.head())

print("Variable cible :")
display(y.head())

# Étape 6 — Réduction de dimension

Lorsqu’un jeu de données contient un très grand nombre de variables, on peut chercher à réduire sa dimension.

Une méthode possible est l’**ACP (Analyse en composantes principales)**.

Cette technique peut permettre de représenter les données avec moins de composantes tout en conservant une partie importante de l’information.

Elle n’est cependant pas nécessaire pour tous les projets.

In [ ]:
from sklearn.decomposition import PCA

# Exemple pédagogique : ACP sur les variables numériques
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X)

print("Nouvelle forme des données :", X_pca.shape)
print("Variance expliquée :", pca.explained_variance_ratio_)

# Étape 7 — Gestion des données déséquilibrées

Dans un problème de classification, certaines classes peuvent être beaucoup plus représentées que d’autres.

Exemple :

- Classe A : 300 observations
- Classe B : 600 observations
- Classe C : 100 observations

La classe C est alors sous-représentée.

Il faut tenir compte de ce déséquilibre lors de la préparation des données et de l’évaluation du modèle.

In [ ]:
# Vérifier la répartition des classes
print("Répartition des classes :")
print(y.value_counts())

print("\nProportions :")
print(y.value_counts(normalize=True))

# Étape 8 — Prétraitement : standardisation

Les variables peuvent avoir des échelles très différentes.

Par exemple :
- nombre de salles de bains : quelques unités ;
- surface d’une maison : plusieurs centaines ou milliers.

La standardisation permet de transformer les variables afin qu’elles soient sur une échelle comparable.

Avec `StandardScaler`, les données sont centrées et réduites.

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

display(X_scaled.head())

# Étape 9 — Séparer les données : entraînement et test

Une partie des données sert à entraîner le modèle et une autre à l’évaluer sur des observations non utilisées pendant l’apprentissage.

Une séparation classique est :
- 80 % entraînement ;
- 20 % test.

Pour une classification, on peut utiliser `stratify=y` afin de conserver une répartition comparable des classes.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

# Étape 10 — Prétraiter correctement après la séparation

Point essentiel :

**Le scaler doit être ajusté uniquement sur les données d’entraînement.**

Ensuite, le même scaler est utilisé pour transformer les données de test.

Cela évite d’utiliser indirectement des informations du jeu de test pendant la préparation du modèle.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Prétraitement terminé.")
print("Le même scaler est utilisé pour train et test.")

# Étape 11 — Validation et stratification

Pour les projets plus sérieux, on peut distinguer :
- ensemble d’entraînement ;
- ensemble de validation ;
- ensemble de test.

L’ensemble de validation permet notamment de comparer différentes configurations et de régler les hyperparamètres.

L’ensemble de test doit rester indépendant pour fournir une évaluation finale.

In [ ]:
# Exemple conceptuel :
#
# Données
# ├── Entraînement
# │   └── utilisé pour apprendre le modèle
# ├── Validation
# │   └── utilisé pour choisir les hyperparamètres
# └── Test
#     └── utilisé pour l'évaluation finale
#
# Dans un projet réel, la validation croisée peut également être utilisée.

# Étape 12 — Choisir plusieurs modèles

Il ne faut pas supposer qu’un modèle complexe sera toujours meilleur.

Selon le problème, on peut comparer :
- régression linéaire ;
- arbre de décision ;
- SVM ;
- forêt aléatoire ;
- réseau neuronal.

Le choix dépend des données, du problème, des ressources disponibles et des performances recherchées.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    "Régression logistique": LogisticRegression(),
    "Arbre de décision": DecisionTreeClassifier(random_state=42),
    "Forêt aléatoire": RandomForestClassifier(random_state=42),
    "SVM": SVC()
}

print("Modèles préparés :", list(models.keys()))

# Étape 13 — Réglage des hyperparamètres

Les hyperparamètres sont des paramètres qui contrôlent le fonctionnement du modèle.

Exemples :
- profondeur maximale d’un arbre ;
- nombre d’arbres dans une forêt aléatoire ;
- paramètres d’un SVM.

On peut rechercher une configuration adaptée avec des méthodes comme `GridSearchCV` ou `RandomizedSearchCV`.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Exemple avec une forêt aléatoire
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [None, 5, 10]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring="accuracy"
)

# Décommenter dans un projet avec suffisamment de données :
# grid.fit(X_train_scaled, y_train)
# print("Meilleurs paramètres :", grid.best_params_)

# Étape 14 — Entraîner le modèle

Une fois le modèle choisi et ses paramètres déterminés, on l’entraîne avec les données d’entraînement.

Dans cet exemple, nous utilisons une régression logistique.

In [ ]:
model = LogisticRegression()

model.fit(X_train_scaled, y_train)

print("Modèle entraîné avec succès.")

# Étape 15 — Faire des prédictions

Le modèle entraîné peut maintenant produire des prédictions sur des données qu’il n’a pas utilisées pendant l’apprentissage.

In [ ]:
y_pred = model.predict(X_test_scaled)

print("Prédictions :")
print(y_pred)

# Étape 16 — Évaluer les performances

Pour une classification, plusieurs métriques peuvent être utilisées :
- accuracy ;
- précision ;
- rappel ;
- score F1.

La métrique doit être choisie en fonction du problème.

Lorsque les classes sont déséquilibrées, l’accuracy seule peut être insuffisante.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy  : {accuracy:.3f}")
print(f"Précision : {precision:.3f}")
print(f"Rappel    : {recall:.3f}")
print(f"F1-score  : {f1:.3f}")

# Étape 17 — Détecter le surapprentissage et le sous-apprentissage

Il faut comparer les performances sur l’entraînement et sur le test.

- Une très bonne performance sur l’entraînement mais une performance beaucoup plus faible sur le test peut indiquer un **surapprentissage**.
- De mauvaises performances sur les deux ensembles peuvent indiquer un **sous-apprentissage**.

Cette analyse doit être interprétée en fonction du modèle et du problème.

In [ ]:
train_accuracy = model.score(X_train_scaled, y_train)
test_accuracy = model.score(X_test_scaled, y_test)

print(f"Performance entraînement : {train_accuracy:.3f}")
print(f"Performance test         : {test_accuracy:.3f}")
print(f"Écart                     : {train_accuracy - test_accuracy:.3f}")

# Étape 18 — Évaluer la robustesse

Une seule division entraînement/test peut donner une estimation particulière des performances.

Pour obtenir une évaluation plus robuste, on peut utiliser la validation croisée.

Elle permet de répéter l’évaluation sur plusieurs divisions des données.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    model,
    X_scaled,
    y,
    cv=5,
    scoring="accuracy"
)

print("Scores :", scores)
print(f"Accuracy moyenne : {scores.mean():.3f}")
print(f"Écart-type       : {scores.std():.3f}")

# Étape 19 — Prendre en compte l’évolution des données

Les données peuvent changer avec le temps.

Dans un problème de prix immobiliers ou de cours boursiers, les relations observées historiquement peuvent évoluer.

Il faut donc vérifier régulièrement si les nouvelles données ressemblent encore aux données utilisées pour entraîner le modèle.

In [ ]:
# Exemple conceptuel de surveillance :
#
# données historiques
#        ↓
# entraînement du modèle
#        ↓
# nouvelles données
#        ↓
# comparaison des distributions
#        ↓
# détection éventuelle d'une dérive
#
# Cette étape nécessite généralement des outils de monitoring
# adaptés au projet réel.

# Étape 20 — Déploiement du modèle

Une fois le modèle validé, il peut être utilisé dans un environnement réel.

Le pipeline doit conserver les mêmes étapes de prétraitement.

Le principe est :

**Nouvelles données → Prétraitement → Modèle → Prédiction**

Il faut notamment conserver le scaler utilisé pendant l’entraînement.

In [ ]:
# Exemple conceptuel pour une nouvelle observation
nouvelle_donnee = pd.DataFrame({
    "age": [30],
    "revenu": [40000]
})

# Utiliser le scaler déjà entraîné
nouvelle_donnee_scaled = scaler.transform(nouvelle_donnee)

prediction = model.predict(nouvelle_donnee_scaled)

print("Prédiction :", prediction[0])

# Étape 21 — Surveillance en production

Après le déploiement, le travail n’est pas terminé.

Il faut surveiller :
- la qualité des nouvelles données ;
- les performances ;
- les erreurs de prédiction ;
- le temps de prédiction ;
- la dérive des données (*data drift*) ;
- la dérive du modèle (*model drift*).

Un modèle peut être performant aujourd’hui et devenir moins adapté lorsque les données évoluent.

In [ ]:
# Checklist de surveillance

monitoring_checklist = [
    "Qualité des nouvelles données",
    "Valeurs manquantes",
    "Évolution des distributions",
    "Performances du modèle",
    "Erreurs de prédiction",
    "Temps de prédiction",
    "Data drift",
    "Model drift"
]

for element in monitoring_checklist:
    print("✓", element)

# Étape 22 — Résumé du pipeline complet

Le pipeline de Machine Learning peut être résumé ainsi :

```text
1. Charger les données
        ↓
2. Comprendre les données
        ↓
3. Nettoyer les données
        ↓
4. Transformer les données
        ↓
5. Réduire éventuellement la dimension
        ↓
6. Gérer les classes déséquilibrées
        ↓
7. Séparer Train / Validation / Test
        ↓
8. Prétraiter les données
        ↓
9. Choisir les modèles
        ↓
10. Régler les hyperparamètres
        ↓
11. Entraîner
        ↓
12. Évaluer
        ↓
13. Vérifier overfitting / underfitting
        ↓
14. Valider la robustesse
        ↓
15. Déployer
        ↓
16. Surveiller data drift / model drift
```

## Idée essentielle

Un projet de Machine Learning ne se limite pas à :

```python
model.fit(X, y)
```

La qualité du résultat dépend de l’ensemble du pipeline, depuis la préparation des données jusqu’à la surveillance du modèle après son déploiement.